# Set up and update the ingestion watermark

Attach `lh_meridian_hr` as the default lakehouse. In Fabric, mark Cell 2 as the parameter cell so the pipeline can override the values when advancing the watermark. Setup mode creates and seeds missing state; update mode advances the selected pipeline watermark.

In [ ]:
mode = "setup"
pipeline_name = "workforce_events"
watermark_timestamp = "2020-12-01 00:00:00"

In [ ]:
from datetime import datetime

if mode not in {"setup", "update"}:
    raise ValueError("mode must be 'setup' or 'update'")
if not pipeline_name.strip():
    raise ValueError("pipeline_name must not be empty")

parsed_watermark = datetime.strptime(watermark_timestamp, "%Y-%m-%d %H:%M:%S")
if parsed_watermark.day != 1:
    raise ValueError("watermark_timestamp must be the first day of a month")

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("""
CREATE TABLE IF NOT EXISTS bronze.ingestion_watermark (
    pipeline_name STRING,
    watermark_timestamp TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
""")

watermark_input = spark.createDataFrame(
    [(pipeline_name, parsed_watermark)],
    "pipeline_name STRING, watermark_timestamp TIMESTAMP",
)
watermark_input.createOrReplaceTempView("watermark_input")

if mode == "update":
    spark.sql("""
    MERGE INTO bronze.ingestion_watermark AS target
    USING watermark_input AS source
    ON target.pipeline_name = source.pipeline_name
    WHEN MATCHED THEN UPDATE SET
        target.watermark_timestamp = source.watermark_timestamp,
        target.updated_at = current_timestamp()
    WHEN NOT MATCHED THEN INSERT (pipeline_name, watermark_timestamp, updated_at)
        VALUES (source.pipeline_name, source.watermark_timestamp, current_timestamp())
    """)
else:
    spark.sql("""
    MERGE INTO bronze.ingestion_watermark AS target
    USING watermark_input AS source
    ON target.pipeline_name = source.pipeline_name
    WHEN NOT MATCHED THEN INSERT (pipeline_name, watermark_timestamp, updated_at)
        VALUES (source.pipeline_name, source.watermark_timestamp, current_timestamp())
    """)

spark.sql("SELECT * FROM bronze.ingestion_watermark ORDER BY pipeline_name").show(truncate=False)